In [ ]:
import os
import subprocess

# 1. SETUP PERCORSI
PATH_EGO_DIR = '/kaggle/input/feature-256/features_256'
PATH_OMNI_DIR = '/kaggle/input/omnivore-features/omnivore'

# 2. CLONAZIONE ANNOTAZIONI (se mancano)
REPO_DIR = '/kaggle/working/annotations'
if not os.path.exists(REPO_DIR):
    print(" Clonazione annotazioni...")
    subprocess.check_call("git clone https://github.com/CaptainCook4D/annotations.git " + REPO_DIR, shell=True)

PATH_CSV = os.path.join(REPO_DIR, 'annotation_csv', 'step_annotations.csv')

# 3. CLONAZIONE CODICE ACTIONFORMER (se manca)
if not os.path.exists('/kaggle/working/actionformer_release'):
    print(" Clonazione ActionFormer...")
    !git clone https://github.com/happyharrycn/actionformer_release.git
else:
    print(" ActionFormer già presente.")

# 4. VERIFICA FILE
def verify(directory, label):
    if os.path.exists(directory):
        print(f" {label}: {len(os.listdir(directory))} file trovati.")
    else:
        print(f" {label}: Cartella NON trovata!")

verify(PATH_EGO_DIR, "EgoVLP")
verify(PATH_OMNI_DIR, "Omnivore")

In [ ]:
import os
import sys
import importlib

# 1. PERCORSI CRITICI
ROOT_DIR = "/kaggle/working/actionformer_release"
UTILS_DIR = os.path.join(ROOT_DIR, "libs", "utils")

# 2. AGGIUNTA FORZATA AL PATH
# Diciamo a Python: "Guarda anche dentro libs/utils quando cerchi moduli!"
if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)
    print(f" Aggiunto {UTILS_DIR} a sys.path")

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)
    print(f" Aggiunto {ROOT_DIR} a sys.path")

# 3. VERIFICA FILE COMPILATO
# Controlliamo se il file .so esiste davvero
files = os.listdir(UTILS_DIR)
so_files = [f for f in files if "nms_1d_cpu" in f and f.endswith(".so")]

if so_files:
    print(f" File compilato trovato: {so_files[0]}")
else:
    print("ERRORE: Non trovo il file .so in libs/utils. La compilazione ha fallito silenziosamente?")

# 4. RESET E IMPORT
os.chdir(ROOT_DIR)
importlib.invalidate_caches()

print("\n Test finale importazione...")
try:
    # Proviamo a importare direttamente il modulo C++
    import nms_1d_cpu
    print("   -> Modulo C++ 'nms_1d_cpu' caricato! ")

    # Proviamo a importare il modello completo
    from libs.modeling.meta_archs import PtTransformer
    print(" SUCCESSO: PtTransformer importato e pronto all'uso!")

except ImportError as e:
    print(f" Errore Importazione: {e}")

In [ ]:
import os
import sys
import subprocess

# 1. Spostiamoci nella cartella dove c'è il file setup.py per le utility
# In ActionFormer di solito è in libs/utils
SETUP_DIR = "/kaggle/working/actionformer_release/libs/utils"

print(f" Avvio compilazione moduli C++ in: {SETUP_DIR}")

if os.path.exists(SETUP_DIR):
    os.chdir(SETUP_DIR)

    try:
        # 2. Eseguiamo il comando di build
        # build_ext --inplace crea il file compilato (.so) direttamente nella cartella corrente
        result = subprocess.check_output(["python", "setup.py", "build_ext", "--inplace"], stderr=subprocess.STDOUT)
        print(result.decode("utf-8")) # Stampa il log della compilazione
        print(" Compilazione completata con successo!")

    except subprocess.CalledProcessError as e:
        print(" Errore durante la compilazione:")
        print(e.output.decode("utf-8"))
else:
    print(f" Cartella non trovata: {SETUP_DIR}")

# 3. Torniamo alla root e riproviamo l'import
ROOT_DIR = "/kaggle/working/actionformer_release"
os.chdir(ROOT_DIR)
print(f"\n Directory di lavoro ripristinata: {os.getcwd()}")

# Aggiungiamo il path se manca
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print("\n Test importazione post-compilazione...")
try:
    import libs
    from libs.modeling.meta_archs import PtTransformer
    print(" SUCCESSO: PtTransformer importato e funzionante!")
except ImportError as e:
    print(f" Ancora errore: {e}")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import random
import time
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW, lr_scheduler
from libs.modeling.meta_archs import PtTransformer

# ==============================================================================
# 1. DATASET SINCRONIZZATO (Fix FPS + Nomi Chiavi)
# ==============================================================================
print(" Rigenerazione Dataset (Sincronizzazione FPS 1.875)...")

PATH_EGO_DIR = '/kaggle/input/feature-256/features_256'
PATH_OMNI_DIR = '/kaggle/input/omnivore-features/omnivore'
PATH_CSV = '/kaggle/working/annotations/annotation_csv/step_annotations.csv'

class SmartRandomCropDataset(Dataset):
    def __init__(self, csv_path, ego_dir, omni_dir, max_len=2304, training=True):
        self.df = pd.read_csv(csv_path)
        self.max_len = max_len
        self.training = training
        self.fps = 1.875  # <--- FIX CRITICO: Allineamento con feature Ego4D
        self.target_dim = 1792

        self.ego_map = self._build_file_map(ego_dir)
        self.omni_map = self._build_file_map(omni_dir)

        self.df['recording_id'] = self.df['recording_id'].astype(str)
        valid_csv_ids = set(self.df['recording_id'].unique())
        common_ids = set(self.ego_map.keys()) & set(self.omni_map.keys())
        self.video_ids = list(common_ids.intersection(valid_csv_ids))
        print(f" Video validi sincronizzati: {len(self.video_ids)}")

    def _build_file_map(self, directory):
        file_map = {}
        if not os.path.exists(directory): return file_map
        for fname in os.listdir(directory):
            if not fname.endswith(('.npz', '.npy')): continue
            vid_id = fname.split('_360p')[0].split('_256')[0].replace('.mp4', '')
            if len(vid_id) > 10 and '_' in vid_id:
                 parts = vid_id.split('_')
                 if len(parts) >= 2 and parts[1].isdigit():
                     vid_id = f"{parts[0]}_{parts[1]}"
            file_map[vid_id] = os.path.join(directory, fname)
        return file_map

    def __len__(self): return len(self.video_ids)

    def __getitem__(self, idx):
        vid_id = self.video_ids[idx]
        try:
            d1 = np.load(self.ego_map[vid_id]); raw_ego = d1[d1.files[0]] if hasattr(d1, 'files') else d1
            d2 = np.load(self.omni_map[vid_id]); raw_omni = d2[d2.files[0]] if hasattr(d2, 'files') else d2

            if raw_ego.shape[0] < raw_ego.shape[1]: raw_ego = raw_ego.T
            if raw_omni.shape[0] < raw_omni.shape[1]: raw_omni = raw_omni.T

            f_ego = torch.from_numpy(raw_ego).float()
            f_omni = torch.from_numpy(raw_omni).float()
            min_t = min(f_ego.shape[0], f_omni.shape[0])
            feat = torch.cat([f_ego[:min_t], f_omni[:min_t]], dim=1)

            if feat.shape[1] < self.target_dim:
                padding = torch.zeros((feat.shape[0], self.target_dim - feat.shape[1]))
                feat = torch.cat([feat, padding], dim=1)

            total_frames = feat.shape[0]
            if self.training and total_frames > self.max_len:
                max_start = total_frames - self.max_len
                start_frame = random.randint(0, max_start)
                end_frame = start_frame + self.max_len
                feat_crop = feat[start_frame:end_frame, :]
                time_offset = start_frame / self.fps
            else:
                feat_crop = feat[:self.max_len, :]
                time_offset = 0.0
                if feat_crop.shape[0] < self.max_len:
                    pad = torch.zeros((self.max_len - feat_crop.shape[0], self.target_dim))
                    feat_crop = torch.cat([feat_crop, pad], dim=0)

            rows = self.df[self.df['recording_id'] == vid_id]
            valid_segments = []
            labels = []
            for _, row in rows.iterrows():
                s, e = float(row['start_time']), float(row['end_time'])
                new_s, new_e = s - time_offset, e - time_offset
                if new_e > 0 and new_s < (self.max_len/self.fps):
                    valid_segments.append([max(0.0, new_s), min(self.max_len/self.fps, new_e)])
                    labels.append(0)

            return {
                'video_id': vid_id,
                'feats': feat_crop.t(),
                'segments': torch.tensor(valid_segments).float() if valid_segments else torch.zeros((0,2)).float(),
                'labels': torch.tensor(labels).long() if labels else torch.zeros((0,)).long(),
                'feat_stride': 1, 'feat_num_frames': feat_crop.shape[0], 'fps': self.fps, 'duration': feat_crop.shape[0]/self.fps
            }
        except: return None

# Istanzia dataset
dataset = SmartRandomCropDataset(PATH_CSV, PATH_EGO_DIR, PATH_OMNI_DIR, training=True)

# ==============================================================================
# 2. TRAINING LOOP AGGRESSIVO
# ==============================================================================
def collate_fn_fix(batch): return [b for b in batch if b is not None]

NUM_EPOCHS = 120
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
CKPT_DIR = '/kaggle/working/checkpoints_180epochs_256'
os.makedirs(CKPT_DIR, exist_ok=True)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_fix, num_workers=2, pin_memory=True)

cfg = {
    "input_dim": 1792, "num_classes": 1, "max_seq_len": 2304,
    "backbone_type": "convTransformer", "backbone_arch": [2, 2, 5],
    "scale_factor": 2, "n_head": 4, "n_mha_win_size": 9,
    "embd_kernel_size": 3, "embd_dim": 512, "embd_with_ln": True,
    "use_abs_pe": True, "use_rel_pe": False, "max_buffer_len_factor": 6.0,
    "fpn_type": "identity", "fpn_dim": 512, "fpn_start_level": 0, "fpn_with_ln": True,
    "head_dim": 512, "head_num_layers": 3, "head_kernel_size": 3, "head_with_ln": True,
    "regression_range": [[0, 4], [4, 8], [8, 16], [16, 32], [32, 64], [64, 10000]],
    "train_cfg": {
        "init_loss_norm": 200,
        "center_sample": "radius",
        "center_sample_radius": 0.7,
        "label_smoothing": 0.0,
        "loss_weight": 2.0,
        "cls_prior_prob": 0.01,
        "dropout": 0.1,
        "droppath": 0.1,
        "head_empty_cls": []
    },
    "test_cfg": {"pre_nms_topk": 5000, "pre_nms_thresh": 0.001, "max_seg_num": 2000, "nms_method": "soft", "iou_threshold": 0.5, "min_score": 0.001, "duration_thresh": 0.05, "multiclass_nms": True, "nms_sigma": 0.50, "voting_thresh": 0.75}
}
print("Inizializzazione Modello...")
model = PtTransformer(**cfg)

# Resume
RESUME_PATH = '/kaggle/input/actionformer-best-epoch-trained-pth/actionformer_best_epoch_trained.pth'
if os.path.exists(RESUME_PATH):
    print(f"RESUME: Caricamento pesi da {RESUME_PATH}")
    state = torch.load(RESUME_PATH)
    clean_state = {k.replace('module.', ''): v for k, v in state.items()} if list(state.keys())[0].startswith('module.') else state
    model.load_state_dict(clean_state)

if torch.cuda.is_available():
    model = model.cuda()
    print(" Training su GPU Singola (Modalità Stabile)")

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"\n AVVIO TRAINING ({NUM_EPOCHS} epoche)...")
print("-" * 60)



# 1. Inizializza le variabili FUORI dal ciclo delle epoche
best_loss = float('inf')
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    # 2. Ciclo di addestramento sui batch
    for i, batch in enumerate(train_loader):
        if not batch: continue
        optimizer.zero_grad()
        outputs = model(batch)
        loss = outputs['final_loss']
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

        if i % 10 == 0:
            print(f"   Ep [{epoch+1}/{NUM_EPOCHS}] Bt [{i}/{len(train_loader)}] Loss: {loss.item():.4f}")

    # 3. Fine Epoca: calcolo media e aggiornamento scheduler
    avg_loss = epoch_loss / len(train_loader)
    scheduler.step()

    print(f" EPOCA {epoch+1} FINITA | Tempo: {time.time()-start_time:.1f}s | Loss: {avg_loss:.4f}")

    # 4. Salvataggio Checkpoint Regolare
    current_ckpt = os.path.join(CKPT_DIR, f"full_epoch_{epoch+1}.pth")
    torch.save(model.state_dict(), current_ckpt)

    # 5. Logica BEST MODEL (confronta con le epoche precedenti)
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_epoch = epoch + 1
        best_path = os.path.join(CKPT_DIR, "best_model_loss.pth")
        torch.save(model.state_dict(), best_path)
        print(f" NUOVO RECORD! Miglior Loss finora: {best_loss:.4f} (Salvato come best_model_loss.pth)")

print(f"\n TRAINING COMPLETATO! La migliore epoca è stata la numero {best_epoch} con loss {best_loss:.4f}")

In [ ]:
import pandas as pd
import torch

def run_inference(model, dataset, output_csv="submission_final.csv"):
    model.eval()
    results = []
    print(f" Avvio Inferenza su {len(dataset)} video...")

    with torch.no_grad():
        for i in range(len(dataset)):
            batch = dataset[i]
            if batch is None: continue

            # Prepariamo le feature [1, C, T] sulla GPU
            feats = batch['feats'].unsqueeze(0).cuda()

            # --- CORREZIONE DEFINITIVA QUI ---
            # ActionFormer richiede TUTTI questi campi per calcolare i segmenti temporali
            video_input = {
                'feats': feats[0],
                'video_id': batch['video_id'],
                'fps': batch['fps'],
                'duration': batch['duration'],
                'feat_stride': batch['feat_stride'],       # <--- MANCAVA QUESTO
                'feat_num_frames': batch['feat_num_frames'] # <--- E QUESTO
            }

            # Passiamo il dizionario completo
            predictions = model([video_input])[0]
            # ---------------------------------

            segs = predictions['segments'].cpu().numpy()
            scores = predictions['scores'].cpu().numpy()

            for j in range(len(segs)):
                results.append({
                    'video_id': batch['video_id'],
                    't_start': round(float(segs[j][0]), 2),
                    't_end': round(float(segs[j][1]), 2),
                    'score': round(float(scores[j]), 4)
                })

            if i % 50 == 0:
                print(f"🔍 Processati {i}/{len(dataset)} video...")

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f" Inferenza completata! File salvato: {output_csv}")

# --- ESECUZIONE ---
# Carichiamo i pesi (assicurati che il path sia giusto)
# Nota: sto usando il path che vedo nel tuo log di errore
best_model_path = "/kaggle/input/best-model-loss-256/best_model_loss_256.pth"

if os.path.exists(best_model_path):
    print(f" Caricamento pesi da: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path))

    # Dataset di test (training=False per evitare random crop)
    test_dataset = SmartRandomCropDataset(PATH_CSV, PATH_EGO_DIR, PATH_OMNI_DIR, training=False)

    run_inference(model, test_dataset)
else:
    print(f" Errore: Il file {best_model_path} non esiste. Controlla il percorso.")

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader

# ==============================================================================
# 1. CONFIGURAZIONE PERCORSI E MODELLO
# ==============================================================================

# Percorsi (Assicurati che siano gli stessi usati nel training)
PATH_EGO_DIR = '/kaggle/input/feature-256/features_256'  # O il path delle tue feature 768
PATH_OMNI_DIR = '/kaggle/input/omnivore-features/omnivore'
PATH_CSV = '/kaggle/working/annotations/annotation_csv/step_annotations.csv'
CHECKPOINT_PATH = '/kaggle/input/best-model-loss-256/best_model_loss_256.pth' # O il tuo path specifico
OUTPUT_CSV = '/kaggle/working/inferenza_finale_256.csv'

# Configurazione Modello (Deve essere identica al training, ma con Soft-NMS per il test)
cfg_inference = {
    "input_dim": 1792, "num_classes": 1, "max_seq_len": 2304,
    "backbone_type": "convTransformer", "backbone_arch": [2, 2, 5],
    "scale_factor": 2, "n_head": 4, "n_mha_win_size": 9,
    "embd_kernel_size": 3, "embd_dim": 512, "embd_with_ln": True,
    "use_abs_pe": True, "use_rel_pe": False, "max_buffer_len_factor": 6.0,
    "fpn_type": "identity", "fpn_dim": 512, "fpn_start_level": 0, "fpn_with_ln": True,
    "head_dim": 512, "head_num_layers": 3, "head_kernel_size": 3, "head_with_ln": True,
    "regression_range": [[0, 4], [4, 8], [8, 16], [16, 32], [32, 64], [64, 10000]],
    "train_cfg": { # Non usata in inferenza ma richiesta per inizializzare
        "init_loss_norm": 200, "center_sample": "radius", "center_sample_radius": 0.7,
        "label_smoothing": 0.0, "loss_weight": 2.0, "cls_prior_prob": 0.01,
        "dropout": 0.1, "droppath": 0.1, "head_empty_cls": []
    },
    # CONFIGURAZIONE TEST (Soft-NMS per massimizzare la Recall)
    "test_cfg": {
        "pre_nms_topk": 5000,
        "pre_nms_thresh": 0.001,
        "max_seg_num": 2000,
        "nms_method": "soft",      # Usa Soft-NMS per risultati migliori
        "iou_threshold": 0.5,      # Soglia più permissiva
        "min_score": 0.001,
        "duration_thresh": 0.05,
        "multiclass_nms": True,
        "nms_sigma": 0.50,
        "voting_thresh": 0.75
    }
}

# ==============================================================================
# 2. FUNZIONE DI INFERENZA (ROBUSTA)
# ==============================================================================
def run_full_inference(model, dataset, output_path):
    model.eval()
    results = []
    print(f" Avvio Inferenza su {len(dataset)} video...")

    with torch.no_grad():
        for i in range(len(dataset)):
            # 1. Carichiamo il dato dal dataset (training=False)
            batch = dataset[i]
            if batch is None: continue

            # 2. Prepariamo l'input per la GPU [Batch=1, Canali, Tempo]
            feats = batch['feats'].unsqueeze(0).cuda()

            # 3. COSTRUIAMO IL DIZIONARIO COMPLETO (Per evitare KeyError)
            # ActionFormer ha bisogno di questi metadati per convertire i bin in secondi
            video_input = {
                'feats': feats[0],
                'video_id': batch['video_id'],
                'fps': batch['fps'],
                'duration': batch['duration'],
                'feat_stride': batch['feat_stride'],
                'feat_num_frames': batch['feat_num_frames']
            }

            # 4. Inferenza
            predictions = model([video_input])[0]

            # 5. Estrazione Risultati
            segs = predictions['segments'].cpu().numpy()
            scores = predictions['scores'].cpu().numpy()

            # 6. Salvataggio nel formato richiesto
            for j in range(len(segs)):
                t_start = float(segs[j][0])
                t_end = float(segs[j][1])
                score = float(scores[j])

                # Filtro di base: scartiamo predizioni con durata <= 0
                if t_end > t_start:
                    results.append({
                        'video_id': batch['video_id'],
                        't_start': round(t_start, 2),
                        't_end': round(t_end, 2),
                        'score': round(score, 4)
                    })

            if i % 50 == 0:
                print(f"Processati {i}/{len(dataset)} video...")

    # Creazione CSV
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False)
    print(f" Inferenza completata! File salvato in: {output_path}")
    print(f" Totale segmenti trovati: {len(df)}")
    return df

# ==============================================================================
# 3. ESECUZIONE
# ==============================================================================

# A. Inizializza il Modello
if 'PtTransformer' not in globals():
    from libs.modeling.meta_archs import PtTransformer # Assicurati che le libs siano importate

model = PtTransformer(**cfg_inference).cuda()

# B. Carica i Pesi
if os.path.exists(CHECKPOINT_PATH):
    print(f" Caricamento pesi da: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH)
    # Gestione compatibilità nomi (rimuove 'module.' se presente)
    clean_ckpt = {k.replace("module.", ""): v for k, v in ckpt.items()}
    model.load_state_dict(clean_ckpt)
else:
    print(f" ATTENZIONE: Checkpoint non trovato in {CHECKPOINT_PATH}")
    # Se non trovi il file, cambia la variabile CHECKPOINT_PATH in alto!

# C. Inizializza il Dataset (training=False è FONDAMENTALE per l'inferenza)
# Nota: Usa la classe SmartRandomCropDataset che hai definito nel blocco precedente
test_dataset = SmartRandomCropDataset(PATH_CSV, PATH_EGO_DIR, PATH_OMNI_DIR, training=False)

# D. Lancia tutto
df_results = run_full_inference(model, test_dataset, OUTPUT_CSV)

# Mostra le prime righe
print("\nAnteprima risultati:")
print(df_results.head())

In [ ]:
import pandas as pd
import numpy as np

# --- CONFIGURAZIONE ---
PATH_GT = '/kaggle/working/annotations/annotation_csv/step_annotations.csv'
PATH_PRED = '/kaggle/working/inferenza_finale_256.csv' # Usa quello pulito, oppure 'submission_final.csv'
IOU_THRESHOLDS = [0.1, 0.3, 0.5, 0.7] # Le soglie da testare (10%, 30%, 50%, 70% di sovrapposizione)

def calculate_iou_1d(pred_start, pred_end, gt_start, gt_end):
    """Calcola la Intersection over Union tra due segmenti temporali."""
    # Intersection
    start_max = max(pred_start, gt_start)
    end_min = min(pred_end, gt_end)
    intersection = max(0, end_min - start_max)

    # Union
    pred_dur = pred_end - pred_start
    gt_dur = gt_end - gt_start
    union = pred_dur + gt_dur - intersection

    if union <= 0: return 0.0
    return intersection / union

def evaluate_recall(gt_path, pred_path, thresholds):
    print(" Caricamento dati...")
    try:
        df_gt = pd.read_csv(gt_path)
        df_pred = pd.read_csv(pred_path)
    except FileNotFoundError as e:
        print(f" Errore: {e}")
        return

    # Pulizia base
    df_pred = df_pred[df_pred['t_end'] > df_pred['t_start']]

    # Dizionario per salvare i risultati
    results = {thresh: 0 for thresh in thresholds}
    total_gt = len(df_gt)

    print(f" Analisi di {total_gt} azioni reali (Ground Truth)...")

    # Raggruppiamo le predizioni per video per velocizzare
    preds_by_video = df_pred.groupby('video_id')

    matches_count = {thresh: 0 for thresh in thresholds}

    # Iteriamo su ogni singola Ground Truth
    for idx, gt_row in df_gt.iterrows():
        vid = gt_row['recording_id']
        gt_s, gt_e = gt_row['start_time'], gt_row['end_time']

        # Se non ci sono predizioni per questo video, la GT è persa
        if vid not in preds_by_video.groups:
            continue

        # Prendiamo le predizioni di quel video
        vid_preds = preds_by_video.get_group(vid)

        # Troviamo la migliore IoU per questa specifica GT
        max_iou = 0.0

        # (Ottimizzazione: controlliamo solo predizioni che hanno senso temporalmente)
        # Un primo filtro grezzo per non calcolare IoU su tutto
        candidates = vid_preds[
            (vid_preds['t_end'] > gt_s) & (vid_preds['t_start'] < gt_e)
        ]

        for _, pred_row in candidates.iterrows():
            iou = calculate_iou_1d(pred_row['t_start'], pred_row['t_end'], gt_s, gt_e)
            if iou > max_iou:
                max_iou = iou

        # Controlliamo quali soglie sono state superate
        for thresh in thresholds:
            if max_iou >= thresh:
                matches_count[thresh] += 1

        if idx % 1000 == 0:
            print(f"   Processati {idx}/{total_gt}...")

    print("\n RISULTATI FINALI (Recall@IoU):")
    print("-" * 40)
    for thresh in thresholds:
        recall = (matches_count[thresh] / total_gt) * 100
        print(f" Recall @ IoU {thresh} ({int(thresh*100)}% overlap): {recall:.2f}%")
        print(f"   (Trovate {matches_count[thresh]} su {total_gt} azioni reali)")
    print("-" * 40)

# ESECUZIONE
evaluate_recall(PATH_GT, PATH_PRED, IOU_THRESHOLDS)